In [ ]:
# In[1]:

import pandas as pd
import pytz
import datetime as dt

# load files (they are in the current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# ensure timestamp is numeric
for df in (metric_df, trace_df, log_df, error_logs_df):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

# timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
win_start = tz.localize(dt.datetime(2021, 3, 10, 16, 30, 0))
win_end = tz.localize(dt.datetime(2021, 3, 10, 17, 0, 0))
win_start_epoch = int(win_start.timestamp())
win_end_epoch = int(win_end.timestamp())

def summarize(df, name, kpi_col=None, kpi_label=None):
    # total rows
    total = int(df.shape[0])
    # distinct cmdb_id sorted, limit display to top 20
    cmdb_vals = sorted(df['cmdb_id'].dropna().astype(str).unique())
    cmdb_display = cmdb_vals if len(cmdb_vals) <= 20 else cmdb_vals[:20]
    # KPI/feature names (if provided)
    if kpi_col and kpi_col in df.columns:
        kpi_vals = sorted(df[kpi_col].dropna().astype(str).unique())
        kpi_display = kpi_vals if len(kpi_vals) <= 20 else kpi_vals[:20]
    else:
        # fallback: for error_logs use 'message' distinct sample; otherwise empty
        if 'message' in df.columns:
            kpi_vals = sorted(df['message'].dropna().astype(str).unique())
            kpi_display = kpi_vals[:20] if len(kpi_vals)>0 else []
            kpi_label = kpi_label or 'message'
        else:
            kpi_vals = []
            kpi_display = []
    # min/max timestamps (epoch and UTC+8 human readable)
    if df['timestamp'].dropna().shape[0] > 0:
        ts_min = int(df['timestamp'].min())
        ts_max = int(df['timestamp'].max())
        hr_min = dt.datetime.fromtimestamp(ts_min, tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        hr_max = dt.datetime.fromtimestamp(ts_max, tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        ts_min = pd.NA
        ts_max = pd.NA
        hr_min = hr_max = None
    # count rows inside incident window (inclusive)
    incident_count = int(df[(df['timestamp'] >= win_start_epoch) & (df['timestamp'] <= win_end_epoch)].shape[0])
    return {
        'file': name,
        'rows': total,
        'cmdb_ids_sample': cmdb_display,
        'cmdb_ids_count': len(cmdb_vals),
        (kpi_label or 'kpi_names'): kpi_display,
        'kpi_count': len(kpi_vals),
        'min_ts_epoch': ts_min,
        'min_ts_local': hr_min,
        'max_ts_epoch': ts_max,
        'max_ts_local': hr_max,
        'incident_window_count': incident_count
    }

summaries = [
    summarize(metric_df, 'metric.csv', kpi_col='kpi_name', kpi_label='kpi_names'),
    summarize(trace_df, 'trace.csv', kpi_col='trace_name', kpi_label='trace_names'),
    summarize(log_df, 'log.csv', kpi_col='log_name', kpi_label='log_names'),
    summarize(error_logs_df, 'error_logs.csv', kpi_col=None, kpi_label='message')
]

summary_df = pd.DataFrame(summaries)

# Keep output compact: show selected columns and limit list display lengths to first 20 items already
summary_df[
    ['file',
     'rows',
     'cmdb_ids_count',
     'cmdb_ids_sample',
     'kpi_count',
     # dynamic KPI column name: get the existing column name for KPI in the df
    ] + [c for c in summary_df.columns if c in ('kpi_names','trace_names','log_names','message')][:1] + [
     'min_ts_epoch','min_ts_local','max_ts_epoch','max_ts_local','incident_window_count']
].head(20), win_start_epoch, win_end_epoch

```
Out[1]:
```
```python
# Build a plain-English summary string from the existing summary_df
def fmt_list_sample(lst):
    if lst is None or (hasattr(lst, '__len__') and len(lst) == 0):
        return "[]"
    # show up to 20 items
    try:
        return str(list(lst)[:20])
    except Exception:
        return str(lst)

rows = []
for _, r in summary_df.iterrows():
    fname = r['file']
    total = int(r['rows'])
    cmdb_count = int(r['cmdb_ids_count'])
    cmdb_sample = fmt_list_sample(r['cmdb_ids_sample'])
    # find KPI-like column for this row
    kpi_col = None
    for col in ('kpi_names','trace_names','log_names','message'):
        if col in summary_df.columns and not pd.isna(r.get(col, pd.NA)):
            kpi_col = col
            break
    kpi_count = int(r['kpi_count']) if 'kpi_count' in r and not pd.isna(r['kpi_count']) else 0
    kpi_sample = fmt_list_sample(r.get(kpi_col, [])) if kpi_col else "[]"
    min_local = r['min_ts_local'] if not pd.isna(r['min_ts_local']) else "N/A"
    max_local = r['max_ts_local'] if not pd.isna(r['max_ts_local']) else "N/A"
    incident_count = int(r['incident_window_count'])
    rows.append(
        f"{fname}: total rows={total}; distinct cmdb_ids={cmdb_count} (sample {cmdb_sample}); "
        f"{'KPI names' if kpi_col!='message' else 'messages'} count={kpi_count} (sample {kpi_sample}); "
        f"time range={min_local} to {max_local}; rows inside incident window={incident_count}."
    )

summary_text = "Telemetry inventory summary (timestamps shown in UTC+8):\n" + "\n".join(rows)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows  cmdb_ids_count                                    cmdb_ids_sample  kpi_count                                          kpi_names min_ts_epoch             min_ts_local max_ts_epoch             max_ts_local  incident_window_count
0      metric.csv  28145              29  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...        370  [Container-DOCKER_CONTAINER_01eeda2c9f0b--bcou...   1615365000  2021-03-10 16:30:00 CST   1615366800  2021-03-10 17:00:00 CST                  28145
1       trace.csv   6813              12  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...         72                                                NaN   1615365000  2021-03-10 16:30:00 CST   1615366740  2021-03-10 16:59:00 CST                   6813
2         log.csv    516               8  [Mysql01, Mysql02, Tomcat01, Tomcat02, Tomcat0...          3                                                NaN   1615365000  2021-03-10 16:30:00 CST   1615366800  2021-03-10 17:00:00 CST                    516
3  error_logs.csv      0               0                                                 []          0                                                NaN         <NA>                     None         <NA>                     None                      0, 1615365000, 1615366800)```
```

In [ ]:
# In[2]:

# Reuse existing metric_df, tz, win_start_epoch, win_end_epoch from previous runs
import pandas as pd
import pytz
import datetime as dt

# defensive: ensure required variables exist and types are correct
tz = pytz.timezone('Asia/Shanghai')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) compute global P95 per (cmdb_id, kpi_name) using full series (no filtering)
group_keys = ['cmdb_id', 'kpi_name']
global_p95 = (
    metric_df
    .groupby(group_keys, observed=True)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 2) total points in full series per group
total_full = (
    metric_df
    .groupby(group_keys, observed=True)
    .size()
    .reset_index(name='total_points_in_full_series')
)

# 3) filter to incident window (inclusive)
window_df = metric_df[(metric_df['timestamp'] >= win_start_epoch) & (metric_df['timestamp'] <= win_end_epoch)].copy()

# 4) total points in window per group
total_window = (
    window_df
    .groupby(group_keys, observed=True)
    .size()
    .reset_index(name='total_points_in_window')
)

# 5) anomalies in window: points with value > global_P95
# merge window points with global_p95 to compare
win_with_p95 = window_df.merge(global_p95, on=group_keys, how='left')
anomalies = win_with_p95[win_with_p95['value'] > win_with_p95['global_P95']].copy()

# aggregate anomalies: count and earliest timestamp
anom_agg = (
    anomalies
    .groupby(group_keys, observed=True)
    .agg(
        anomalies_in_window_count=('value','size'),
        earliest_anom_ts_epoch=('timestamp','min')
    )
    .reset_index()
)

# convert earliest_anom_ts_epoch to localized human-readable string
def epoch_to_local_str(e):
    if pd.isna(e):
        return pd.NA
    try:
        return dt.datetime.fromtimestamp(int(e), tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return pd.NA

if not anom_agg.empty:
    anom_agg['earliest_anomaly_ts_local'] = anom_agg['earliest_anom_ts_epoch'].apply(epoch_to_local_str)
else:
    anom_agg['earliest_anomaly_ts_local'] = pd.Series(dtype="object")

# 6) max value in window per group (even if zero rows then missing)
max_in_window = (
    window_df
    .groupby(group_keys, observed=True)['value']
    .max()
    .reset_index(name='max_value_in_window')
)

# 7) assemble summary: start from global_p95 (covers all groups present in full series)
summary = global_p95.merge(total_full, on=group_keys, how='left')
summary = summary.merge(total_window, on=group_keys, how='left')
summary = summary.merge(anom_agg[[*group_keys, 'anomalies_in_window_count', 'earliest_anomaly_ts_local']], on=group_keys, how='left')
summary = summary.merge(max_in_window, on=group_keys, how='left')

# fill NaNs for counts as 0
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomalies_in_window_count'] = summary['anomalies_in_window_count'].fillna(0).astype(int)
summary['earliest_anomaly_ts_local'] = summary['earliest_anomaly_ts_local'].fillna(pd.NA)

# keep requested columns and sort by anomalies descending, limit to top 20
result_df = (
    summary[
        ['cmdb_id','kpi_name','global_P95','total_points_in_full_series',
         'total_points_in_window','anomalies_in_window_count',
         'earliest_anomaly_ts_local','max_value_in_window']
    ]
    .sort_values(by='anomalies_in_window_count', ascending=False)
    .reset_index(drop=True)
    .head(20)
)

# keep output compact: round global_P95 and max_value_in_window for readability
result_df['global_P95'] = result_df['global_P95'].round(6)
result_df['max_value_in_window'] = result_df['max_value_in_window'].round(6)

result_df

```
Out[2]:
```
# Build a plain-English summary string from the previously computed result_df
import pandas as pd
import math

def fmt_num(v):
    if pd.isna(v):
        return "N/A"
    try:
        v = float(v)
    except Exception:
        return str(v)
    if abs(v) >= 1000:
        return f"{v:,.0f}"
    # small numbers: show up to 6 decimal places, trim trailing zeros
    return f"{v:.6f}".rstrip('0').rstrip('.')

# ensure result_df exists
summary_df = result_df.copy()

# compute counts
num_rows = summary_df.shape[0]
num_with_anom = int((summary_df['anomalies_in_window_count'] > 0).sum())

lines = []
lines.append("Summary of metric anomalies during 2021-03-10 16:30:00 to 2021-03-10 17:00:00 (UTC+8):")
lines.append(f"- Total component-KPI rows returned (top candidates): {num_rows}")
lines.append(f"- Component-KPI pairs with >0 points above their global P95 in the window: {num_with_anom}")
lines.append("- Top findings (showing up to 20 rows):")

for i, row in summary_df.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    gp95 = fmt_num(row['global_P95'])
    total_full = int(row['total_points_in_full_series'])
    total_win = int(row['total_points_in_window'])
    anom_count = int(row['anomalies_in_window_count'])
    earliest = row['earliest_anomaly_ts_local'] if pd.notna(row['earliest_anomaly_ts_local']) else "N/A"
    max_win = fmt_num(row['max_value_in_window'])
    lines.append(f"{i+1}) {cmdb} - {kpi}: global_P95={gp95}; full_points={total_full}; window_points={total_win}; "
                 f"anomalies_in_window={anom_count}; earliest_anomaly={earliest}; max_in_window={max_win}")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_P95  total_points_in_full_series  total_points_in_window  anomalies_in_window_count earliest_anomaly_ts_local  max_value_in_window
0    Redis01  redis-Redis_6379_Redis  (total_connections_rec...  3.264243e+07                           30                      30                          2   2021-03-10 16:58:00 CST         3.264301e+07
1    Redis01              redis-Redis_6379_Redis  (used_memory)  1.812164e+07                           30                      30                          2   2021-03-10 16:50:00 CST         1.852694e+07
2   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  3.748908e+08                           31                      31                          2   2021-03-10 16:59:00 CST         3.753038e+08
3   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  4.932825e+05                           31                      31                          2   2021-03-10 16:59:00 CST         4.940030e+05
4    Redis01            redis-Redis_6379_Redis  (used_cpu_user)  3.478109e+04                           30                      30                          2   2021-03-10 16:58:00 CST         3.478319e+04
5    Redis01            redis-Redis_6379_Redis  (keyspace_hits)  3.940208e+09                           30                      30                          2   2021-03-10 16:58:00 CST         3.940580e+09
6    Redis01  redis-Redis_6379_Redis  (instantaneous_ops_per...  4.978950e+03                           30                      30                          2   2021-03-10 16:48:00 CST         5.905000e+03
7    Redis01  redis-Redis_6379_Redis  (total_commands_proces...  5.178145e+09                           30                      30                          2   2021-03-10 16:58:00 CST         5.178598e+09
8    Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las...  1.140305e+09                           30                      30                          2   2021-03-10 16:58:00 CST         1.140385e+09
9   apache01                         OSLinux-CPU_CPU_CPUSysTime  1.855935e+01                           31                      31                          2   2021-03-10 16:33:00 CST         1.868000e+01
10  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  7.500000e+00                           31                      31                          2   2021-03-10 16:30:00 CST         1.100000e+01
11   Redis01                redis-Redis_6379_Redis  (lru_clock)  4.753918e+06                           30                      30                          2   2021-03-10 16:58:00 CST         4.754005e+06
12   Redis02                        OSLinux-CPU_CPU_CPUidleutil  9.903254e+01                           31                      31                          2   2021-03-10 16:43:00 CST         9.911158e+01
13   Redis02                             OSLinux-CPU_CPU_CPUWio  1.399500e-01                           31                      31                          2   2021-03-10 16:37:00 CST         1.759090e-01
14   Redis02                        OSLinux-CPU_CPU_CPUUserTime  4.160500e-01                           31                      31                          2   2021-03-10 16:42:00 CST         4.263140e-01
15   Redis02                         OSLinux-CPU_CPU_CPUSysTime  1.000150e+00                           31                      31                          2   2021-03-10 16:50:00 CST         1.074145e+00
16   Mysql01                         OSLinux-CPU_CPU_CPUSysTime  5.111580e-01                           31                      31                          2   2021-03-10 16:34:00 CST         5.198000e-01
17   Mysql01                            OSLinux-CPU_CPU_CPULoad  2.850000e-01                           31                      31                          2   2021-03-10 16:46:00 CST         4.500000e-01
18   Mysql01                        OSLinux-CPU_CPU_CPUidleutil  9.872985e+01                           31                      31                          2   2021-03-10 16:42:00 CST         9.883605e+01
19   Mysql01                             OSLinux-CPU_CPU_CPUWio  1.691550e+00                           31                      31                          2   2021-03-10 16:33:00 CST         1.912800e+00```
```

In [ ]:
# In[3]:

# Reuse metric_df, result_df, tz, win_start_epoch, win_end_epoch from previous cells
import pandas as pd
import datetime as dt

# Select component-KPI pairs with anomalies_in_window_count > 0
anomaly_pairs = result_df[result_df['anomalies_in_window_count'] > 0][['cmdb_id','kpi_name','global_P95']].copy()

# If no pairs, return empty DataFrames
if anomaly_pairs.empty:
    fault_segments_top = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_P95','fault_start_ts_local','fault_end_ts_local',
        'duration_minutes','max_value_in_segment','severity_pct'
    ])
    top_cmdb_summary = pd.DataFrame(columns=['cmdb_id','number_of_segments'])
else:
    # Merge to get full series for these pairs (global_P95 attached from anomaly_pairs)
    merged = metric_df.merge(anomaly_pairs, on=['cmdb_id','kpi_name'], how='inner', suffixes=('','_p95'))
    # Ensure numeric types
    merged['timestamp'] = pd.to_numeric(merged['timestamp'], errors='coerce').astype('Int64')
    merged['value'] = pd.to_numeric(merged['value'], errors='coerce')

    # Filter to incident window
    window = merged[(merged['timestamp'] >= win_start_epoch) & (merged['timestamp'] <= win_end_epoch)].copy()

    # Mark anomaly points where value > global_P95 (use the global_P95 from anomaly_pairs)
    window['is_anom'] = window['value'] > window['global_P95']

    # Keep only anomaly points for segmentation
    anoms = window[window['is_anom']].copy()

    if anoms.empty:
        fault_segments_top = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','global_P95','fault_start_ts_local','fault_end_ts_local',
            'duration_minutes','max_value_in_segment','severity_pct'
        ])
        top_cmdb_summary = pd.DataFrame(columns=['cmdb_id','number_of_segments'])
    else:
        # Assign consecutive-minute segment ids per (cmdb_id, kpi_name)
        def assign_segments(df):
            df = df.sort_values('timestamp').copy()
            # large value for first diff to ensure it's treated as new segment
            df['gap'] = df['timestamp'].diff().fillna(999999)
            df['new_seg_flag'] = (df['gap'] != 60).astype(int)
            df['seg_idx'] = df['new_seg_flag'].cumsum()
            return df

        anoms_seg = anoms.groupby(['cmdb_id','kpi_name'], observed=True, group_keys=False).apply(assign_segments).reset_index(drop=True)

        # Aggregate segments
        agg = (
            anoms_seg
            .groupby(['cmdb_id','kpi_name','seg_idx'], observed=True, as_index=False)
            .agg(
                global_P95=('global_P95','first'),
                fault_start_ts_epoch=('timestamp','min'),
                fault_end_ts_epoch=('timestamp','max'),
                points_in_segment=('timestamp','size'),
                max_value_in_segment=('value','max')
            )
        )

        # Compute duration in inclusive minutes and severity_pct
        agg['duration_minutes'] = ((agg['fault_end_ts_epoch'] - agg['fault_start_ts_epoch']) / 60).astype(int) + 1
        # safety for division by zero
        def compute_severity(maxv, p95):
            try:
                if pd.isna(p95) or float(p95) == 0:
                    return float('nan')
                return (float(maxv) - float(p95)) / float(p95) * 100.0
            except Exception:
                return float('nan')
        agg['severity_pct'] = agg.apply(lambda r: compute_severity(r['max_value_in_segment'], r['global_P95']), axis=1)

        # Convert epochs to localized human-readable strings (UTC+8)
        def epoch_to_local(e):
            return dt.datetime.fromtimestamp(int(e), tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        agg['fault_start_ts_local'] = agg['fault_start_ts_epoch'].apply(epoch_to_local)
        agg['fault_end_ts_local'] = agg['fault_end_ts_epoch'].apply(epoch_to_local)

        # Build final segments table with required columns
        segments_df = agg[[
            'cmdb_id','kpi_name','global_P95','fault_start_ts_local','fault_end_ts_local',
            'duration_minutes','max_value_in_segment','severity_pct'
        ]].copy()

        # Sort by severity_pct descending and keep top 20
        fault_segments_top = segments_df.sort_values(by='severity_pct', ascending=False).head(20).reset_index(drop=True)

        # Per-cmdb_id counts of segments (and top 10)
        top_cmdb_summary = (
            segments_df
            .groupby('cmdb_id', observed=True)
            .size()
            .reset_index(name='number_of_segments')
            .sort_values(by='number_of_segments', ascending=False)
            .head(10)
            .reset_index(drop=True)
        )

# Display compact results (variables shown to IPython)
fault_segments_top, top_cmdb_summary

```
Out[3]:
```
# Build a plain-English summary from the previously computed tables
import pandas as pd

# Use existing variables from the session: fault_segments_top, top_cmdb_summary
# Defensive checks
if 'fault_segments_top' not in globals() or 'top_cmdb_summary' not in globals():
    summary_text = "Required results are not available in the current session."
else:
    segs = fault_segments_top.copy()
    cmdb_summary = top_cmdb_summary.copy()
    # compact totals
    total_reported_segments = int(cmdb_summary['number_of_segments'].sum()) if not cmdb_summary.empty else 0
    num_segments_shown = segs.shape[0]
    # prepare top segments (up to 5) with concise fields
    top_n = min(5, num_segments_shown)
    lines = []
    lines.append(f"Fault segment analysis summary (incident window 2021-03-10 16:30–17:00 UTC+8):")
    lines.append(f"- Total segments reported (top CMDB summary rows): {total_reported_segments}")
    lines.append(f"- Number of fault-segment rows returned (displayed up to 20): {num_segments_shown}")
    lines.append("")
    lines.append("Top CMDBs by number of segments (top entries):")
    for _, r in cmdb_summary.head(10).iterrows():
        lines.append(f"  - {r['cmdb_id']}: {int(r['number_of_segments'])} segments")
    lines.append("")
    lines.append(f"Top {top_n} fault segments by severity (highest severity first):")
    for i in range(top_n):
        r = segs.loc[i]
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        gp95 = float(r['global_P95']) if pd.notna(r['global_P95']) else None
        start = r['fault_start_ts_local']
        end = r['fault_end_ts_local']
        dur = int(r['duration_minutes'])
        maxv = float(r['max_value_in_segment']) if pd.notna(r['max_value_in_segment']) else None
        sev = float(r['severity_pct']) if pd.notna(r['severity_pct']) else None
        lines.append(
            f"  {i+1}) {cmdb} | {kpi} | severity={sev:.3f}% | period={start} to {end} "
            f"({dur} min) | max={maxv:.6g} | global_P95={gp95:.6g}"
        )
    lines.append("")
    lines.append("Quick interpretation:")
    lines.append("- Redis01, Redis02 and Mysql01 show the largest number of fault segments during the window (Redis01=10, Redis02=8, Mysql01=7).")
    lines.append("- The single most severe segment is a CPU load spike on Mysql01 (≈57.9% above its global P95), followed by a Tomcat threads spike and several Redis and OS CPU-related events.")
    lines.append("- Most segments are short (1–2 minutes) and are concentrated on Redis, MySQL and Tomcat components during the incident window.")
    summary_text = "\n".join(lines)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    global_P95     fault_start_ts_local       fault_end_ts_local  duration_minutes  max_value_in_segment  severity_pct
0    Mysql01                            OSLinux-CPU_CPU_CPULoad  2.850000e-01  2021-03-10 16:46:00 CST  2021-03-10 16:47:00 CST                 2          4.500000e-01     57.894737
1   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  7.500000e+00  2021-03-10 16:30:00 CST  2021-03-10 16:30:00 CST                 1          1.100000e+01     46.666667
2    Redis02                             OSLinux-CPU_CPU_CPUWio  1.399500e-01  2021-03-10 16:37:00 CST  2021-03-10 16:37:00 CST                 1          1.759090e-01     25.694176
3    Redis01  redis-Redis_6379_Redis  (instantaneous_ops_per...  4.978950e+03  2021-03-10 16:48:00 CST  2021-03-10 16:48:00 CST                 1          5.905000e+03     18.599303
4    Mysql01                             OSLinux-CPU_CPU_CPUWio  1.691550e+00  2021-03-10 16:33:00 CST  2021-03-10 16:33:00 CST                 1          1.912800e+00     13.079720
5    Redis02                             OSLinux-CPU_CPU_CPUWio  1.399500e-01  2021-03-10 16:56:00 CST  2021-03-10 16:56:00 CST                 1          1.505000e-01      7.538407
6    Redis02                         OSLinux-CPU_CPU_CPUSysTime  1.000150e+00  2021-03-10 17:00:00 CST  2021-03-10 17:00:00 CST                 1          1.074145e+00      7.398390
7   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  7.500000e+00  2021-03-10 16:50:00 CST  2021-03-10 16:50:00 CST                 1          8.000000e+00      6.666667
8    Redis02                        OSLinux-CPU_CPU_CPUUserTime  4.160500e-01  2021-03-10 17:00:00 CST  2021-03-10 17:00:00 CST                 1          4.263140e-01      2.467011
9    Redis02                        OSLinux-CPU_CPU_CPUUserTime  4.160500e-01  2021-03-10 16:42:00 CST  2021-03-10 16:42:00 CST                 1          4.263000e-01      2.463646
10   Redis01              redis-Redis_6379_Redis  (used_memory)  1.812164e+07  2021-03-10 16:50:00 CST  2021-03-10 16:50:00 CST                 1          1.852694e+07      2.236528
11   Mysql01                         OSLinux-CPU_CPU_CPUSysTime  5.111580e-01  2021-03-10 16:34:00 CST  2021-03-10 16:34:00 CST                 1          5.198000e-01      1.690671
12   Redis01  redis-Redis_6379_Redis  (instantaneous_ops_per...  4.978950e+03  2021-03-10 16:59:00 CST  2021-03-10 16:59:00 CST                 1          5.037000e+03      1.165908
13   Mysql01                             OSLinux-CPU_CPU_CPUWio  1.691550e+00  2021-03-10 16:57:00 CST  2021-03-10 16:57:00 CST                 1          1.708100e+00      0.978393
14  apache01                         OSLinux-CPU_CPU_CPUSysTime  1.855935e+01  2021-03-10 16:37:00 CST  2021-03-10 16:37:00 CST                 1          1.868000e+01      0.650077
15   Redis02                         OSLinux-CPU_CPU_CPUSysTime  1.000150e+00  2021-03-10 16:50:00 CST  2021-03-10 16:50:00 CST                 1          1.004100e+00      0.394941
16   Redis01              redis-Redis_6379_Redis  (used_memory)  1.812164e+07  2021-03-10 16:58:00 CST  2021-03-10 16:58:00 CST                 1          1.815919e+07      0.207220
17  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  4.932825e+05  2021-03-10 16:59:00 CST  2021-03-10 17:00:00 CST                 2          4.940030e+05      0.146062
18  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  3.748908e+08  2021-03-10 16:59:00 CST  2021-03-10 17:00:00 CST                 2          3.753038e+08      0.110166
19   Mysql01                        OSLinux-CPU_CPU_CPUidleutil  9.872985e+01  2021-03-10 16:44:00 CST  2021-03-10 16:44:00 CST                 1          9.883605e+01      0.107565,     cmdb_id  number_of_segments
0   Redis01                  10
1   Redis02                   8
2   Mysql01                   7
3  Tomcat03                   4
4  apache01                   2)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import datetime as dt
import numpy as np

# reuse existing variables if available; otherwise load trace.csv
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure types and timezone
tz = pytz.timezone('Asia/Shanghai')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# incident window epochs (UTC+8)
win_start = tz.localize(dt.datetime(2021,3,10,16,30,0))
win_end   = tz.localize(dt.datetime(2021,3,10,17,0,0))
win_start_epoch = int(win_start.timestamp())
win_end_epoch = int(win_end.timestamp())

# 1) compute global P95 per (cmdb_id, trace_name) using full series (no filtering)
group_keys = ['cmdb_id','trace_name']
global_p95 = (
    trace_df
    .groupby(group_keys, observed=True)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 2) total points in full series per group
total_full = (
    trace_df
    .groupby(group_keys, observed=True)
    .size()
    .reset_index(name='total_points_in_full_series')
)

# 3) filter to incident window (inclusive)
window_df = trace_df[(trace_df['timestamp'] >= win_start_epoch) & (trace_df['timestamp'] <= win_end_epoch)].copy()

# 4) total points in window per group
total_window = (
    window_df
    .groupby(group_keys, observed=True)
    .size()
    .reset_index(name='total_points_in_window')
)

# 5) anomalies in window: value > global_P95
win_with_p95 = window_df.merge(global_p95, on=group_keys, how='left')

anomalies = win_with_p95[win_with_p95['value'] > win_with_p95['global_P95']].copy()

# aggregate anomalies: count and earliest timestamp
anom_agg = (
    anomalies
    .groupby(group_keys, observed=True)
    .agg(
        anomalies_in_window_count=('value','size'),
        earliest_anom_ts_epoch=('timestamp','min')
    )
    .reset_index()
)

# convert earliest_anom_ts_epoch to localized human-readable string
def epoch_to_local_str(e):
    if pd.isna(e):
        return pd.NA
    try:
        return dt.datetime.fromtimestamp(int(e), tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return pd.NA

if not anom_agg.empty:
    anom_agg['earliest_anomaly_ts_local'] = anom_agg['earliest_anom_ts_epoch'].apply(epoch_to_local_str)
else:
    anom_agg['earliest_anomaly_ts_local'] = pd.Series(dtype="object")

# 6) max value in window per group
max_in_window = (
    window_df
    .groupby(group_keys, observed=True)['value']
    .max()
    .reset_index(name='max_value_in_window')
)

# 7) assemble summary
summary = global_p95.merge(total_full, on=group_keys, how='left')
summary = summary.merge(total_window, on=group_keys, how='left')
summary = summary.merge(anom_agg[[*group_keys, 'anomalies_in_window_count', 'earliest_anomaly_ts_local']], on=group_keys, how='left')
summary = summary.merge(max_in_window, on=group_keys, how='left')

# fill NaNs for counts
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomalies_in_window_count'] = summary['anomalies_in_window_count'].fillna(0).astype(int)
summary['earliest_anomaly_ts_local'] = summary['earliest_anomaly_ts_local'].fillna(pd.NA)

# 8) compute sorting metric: (max_value_in_window/global_P95 - 1)
def safe_ratio(maxv, p95):
    try:
        if pd.isna(maxv) or pd.isna(p95) or float(p95) == 0:
            return np.nan
        return float(maxv) / float(p95) - 1.0
    except Exception:
        return np.nan

summary['max_over_p95_minus1'] = summary.apply(lambda r: safe_ratio(r['max_value_in_window'], r['global_P95']), axis=1)

# 9) final selection and sorting: anomalies desc, then ratio desc
result_df = (
    summary[
        ['cmdb_id','trace_name','global_P95','total_points_in_full_series',
         'total_points_in_window','anomalies_in_window_count',
         'earliest_anomaly_ts_local','max_value_in_window','max_over_p95_minus1']
    ]
    .sort_values(by=['anomalies_in_window_count','max_over_p95_minus1'], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# Round numeric columns for compact display
result_df['global_P95'] = result_df['global_P95'].round(6)
result_df['max_value_in_window'] = result_df['max_value_in_window'].round(6)
result_df['max_over_p95_minus1'] = result_df['max_over_p95_minus1'].round(6)

# Keep output compact: show only requested columns (excluding helper ratio if not needed)
final_df = result_df[[
    'cmdb_id','trace_name','global_P95','total_points_in_full_series',
    'total_points_in_window','anomalies_in_window_count','earliest_anomaly_ts_local','max_value_in_window'
]]

final_df

```
Out[4]:
```
# Summarize the trace.csv anomaly table previously computed (final_df)
import pandas as pd
import numpy as np

# Use final_df from previous computation; defensive fallback to reload if missing
try:
    df = final_df.copy()
except NameError:
    df = pd.read_csv("trace.csv")  # fallback (shouldn't be needed)

# ensure numeric
df['global_P95'] = pd.to_numeric(df['global_P95'], errors='coerce')
df['max_value_in_window'] = pd.to_numeric(df['max_value_in_window'], errors='coerce')

# basic counts
n_rows = int(df.shape[0])
unique_cmdb = sorted(df['cmdb_id'].dropna().unique())
n_cmdb = len(unique_cmdb)

# compute ratio and sort to find top spikes
df['factor_over_p95'] = df.apply(lambda r: (float(r['max_value_in_window'])/float(r['global_P95'])) if pd.notna(r['global_P95']) and r['global_P95']!=0 else np.nan, axis=1)
top_spikes = df.sort_values('factor_over_p95', ascending=False).head(3)

# Build summary string
lines = []
lines.append("Trace anomaly summary (incident window 2021-03-10 16:30:00 to 2021-03-10 17:00:00, UTC+8):")
lines.append(f"- Rows returned (top 20): {n_rows}; distinct cmdb_ids in this set: {n_cmdb} ({', '.join(unique_cmdb[:10])})")
lines.append("- Method: global P95 computed per (cmdb_id, trace_name) on full series; anomalies = points in window with value > global_P95.")
lines.append("")
lines.append("Key observations:")
lines.append("- All top rows show 2 anomaly points during the window and 30 points in the window/full series for these traces.")
lines.append("- Earliest anomaly for many pairs is at 2021-03-10 16:30:00 (UTC+8).")
lines.append("")
lines.append("Top relative spikes (max / global_P95):")
for _, r in top_spikes.iterrows():
    cmdb = r['cmdb_id']
    trace = r['trace_name']
    p95 = r['global_P95']
    mx = r['max_value_in_window']
    factor = r['factor_over_p95']
    earliest = r['earliest_anomaly_ts_local'] if 'earliest_anomaly_ts_local' in r and pd.notna(r['earliest_anomaly_ts_local']) else "2021-03-10 16:30:00 CST"
    lines.append(f" - {cmdb} | {trace}: global_P95={p95:.6g}, max_in_window={mx:.6g}, factor≈{factor:.2f}x, earliest_anomaly={earliest}")

lines.append("")
lines.append("Interpretation:")
lines.append("- The top anomalies are latency spikes on traces between dockerA1/A2/B2 and MG01/MG02 (both duration_mean and duration_p95 metrics).")
lines.append("- Example: MG02 trace.from_dockerA1.duration_mean peaked ~2.00s vs P95 ~0.123s (~16× above P95); duration_p95 series peaked ≈9.46s vs P95 ≈1.01s (~9×).")
lines.append("- These consistent, high-magnitude trace latency spikes starting at 16:30 suggest a service-to-service latency issue (network or downstream processing) affecting MG01/MG02 entry points and the dockerA clients during the incident window.")
lines.append("")
lines.append("Recommendation: prioritize investigation of MG01/MG02 and the dockerA* -> MG* call paths (network and downstream service performance) for root-cause analysis.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  global_P95  total_points_in_full_series  total_points_in_window  anomalies_in_window_count earliest_anomaly_ts_local  max_value_in_window
0       MG02  trace.from_dockerA1.duration_mean    0.122864                           30                      30                          2   2021-03-10 16:30:00 CST             2.003846
1   dockerA1        trace.to_MG02.duration_mean    0.122864                           30                      30                          2   2021-03-10 16:30:00 CST             2.003846
2       MG01  trace.from_dockerA1.duration_mean    0.123705                           30                      30                          2   2021-03-10 16:30:00 CST             1.564438
3   dockerA1        trace.to_MG01.duration_mean    0.123705                           30                      30                          2   2021-03-10 16:30:00 CST             1.564438
4       MG02  trace.from_dockerA2.duration_mean    0.116432                           30                      30                          2   2021-03-10 16:30:00 CST             1.463302
5   dockerA2        trace.to_MG02.duration_mean    0.116432                           30                      30                          2   2021-03-10 16:30:00 CST             1.463302
6       MG01  trace.from_dockerB2.duration_mean    0.107985                           30                      30                          2   2021-03-10 16:30:00 CST             1.016439
7   dockerB2        trace.to_MG01.duration_mean    0.107985                           30                      30                          2   2021-03-10 16:30:00 CST             1.016439
8       MG02   trace.from_dockerA1.duration_p95    1.013680                           30                      30                          2   2021-03-10 16:30:00 CST             9.458700
9   dockerA1         trace.to_MG02.duration_p95    1.013680                           30                      30                          2   2021-03-10 16:30:00 CST             9.458700
10      MG02   trace.from_dockerA2.duration_p95    1.025158                           30                      30                          2   2021-03-10 16:30:00 CST             8.509700
11  dockerA2         trace.to_MG02.duration_p95    1.025158                           30                      30                          2   2021-03-10 16:30:00 CST             8.509700
12      MG01   trace.from_dockerA2.duration_p95    1.043600                           30                      30                          2   2021-03-10 16:30:00 CST             8.029750
13  dockerA2         trace.to_MG01.duration_p95    1.043600                           30                      30                          2   2021-03-10 16:30:00 CST             8.029750
14      MG01   trace.from_dockerA1.duration_p95    1.032425                           30                      30                          2   2021-03-10 16:30:00 CST             7.416400
15  dockerA1         trace.to_MG01.duration_p95    1.032425                           30                      30                          2   2021-03-10 16:30:00 CST             7.416400
16      MG01   trace.from_dockerB2.duration_p95    1.040502                           30                      30                          2   2021-03-10 16:30:00 CST             7.391350
17  dockerB2         trace.to_MG01.duration_p95    1.040502                           30                      30                          2   2021-03-10 16:30:00 CST             7.391350
18      MG02  trace.from_dockerB2.duration_mean    0.114665                           30                      30                          2   2021-03-10 16:30:00 CST             0.794688
19  dockerB2        trace.to_MG02.duration_mean    0.114665                           30                      30                          2   2021-03-10 16:30:00 CST             0.794688```
```

In [ ]:
# In[5]:

# Reuse trace_df, tz, win_start_epoch, win_end_epoch if available; otherwise (re)define
import pandas as pd
import pytz
import datetime as dt
import numpy as np

try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')
win_start = tz.localize(dt.datetime(2021,3,10,16,30,0))
win_end   = tz.localize(dt.datetime(2021,3,10,17,0,0))
win_start_epoch = int(win_start.timestamp())
win_end_epoch = int(win_end.timestamp())

# 1) Identify trace_name values containing 'mysql' or 'redis' (case-insensitive)
mask_names = trace_df['trace_name'].fillna('').str.contains('mysql|redis', case=False, na=False)
matched_df = trace_df[mask_names].copy()

# If none matched, return empty DataFrame with required columns
if matched_df.empty:
    cols = ['cmdb_id','trace_name','global_P95','total_points_in_full_series',
            'total_points_in_window','anomalies_in_window_count','earliest_anomaly_ts_local','max_value_in_window']
    final_df = pd.DataFrame(columns=cols)
else:
    group_keys = ['cmdb_id','trace_name']
    # 2) Compute global P95 per group using full series of matched groups (no window filter)
    global_p95 = (
        matched_df
        .groupby(group_keys, observed=True)['value']
        .quantile(0.95)
        .reset_index(name='global_P95')
    )

    # 3) total points in full series per group
    total_full = (
        matched_df
        .groupby(group_keys, observed=True)
        .size()
        .reset_index(name='total_points_in_full_series')
    )

    # 4) filter matched series to incident window (inclusive)
    window_df = matched_df[(matched_df['timestamp'] >= win_start_epoch) & (matched_df['timestamp'] <= win_end_epoch)].copy()

    # 5) total points in window per group
    total_window = (
        window_df
        .groupby(group_keys, observed=True)
        .size()
        .reset_index(name='total_points_in_window')
    )

    # 6) merge window points with global_p95 to identify anomalies
    win_with_p95 = window_df.merge(global_p95, on=group_keys, how='left')

    anomalies = win_with_p95[win_with_p95['value'] > win_with_p95['global_P95']].copy()

    # 7) aggregate anomalies: count and earliest timestamp
    anom_agg = (
        anomalies
        .groupby(group_keys, observed=True)
        .agg(
            anomalies_in_window_count=('value','size'),
            earliest_anom_ts_epoch=('timestamp','min')
        )
        .reset_index()
    )

    # convert earliest epoch to localized string
    def epoch_to_local_str(e):
        if pd.isna(e):
            return pd.NA
        try:
            return dt.datetime.fromtimestamp(int(e), tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        except Exception:
            return pd.NA

    if not anom_agg.empty:
        anom_agg['earliest_anomaly_ts_local'] = anom_agg['earliest_anom_ts_epoch'].apply(epoch_to_local_str)
    else:
        anom_agg['earliest_anomaly_ts_local'] = pd.Series(dtype="object")

    # 8) max value in window per group
    max_in_window = (
        window_df
        .groupby(group_keys, observed=True)['value']
        .max()
        .reset_index(name='max_value_in_window')
    )

    # 9) assemble summary table
    summary = global_p95.merge(total_full, on=group_keys, how='left')
    summary = summary.merge(total_window, on=group_keys, how='left')
    summary = summary.merge(anom_agg[[*group_keys, 'anomalies_in_window_count', 'earliest_anomaly_ts_local']], on=group_keys, how='left')
    summary = summary.merge(max_in_window, on=group_keys, how='left')

    # fill NaNs for counts
    summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
    summary['anomalies_in_window_count'] = summary['anomalies_in_window_count'].fillna(0).astype(int)
    summary['earliest_anomaly_ts_local'] = summary['earliest_anomaly_ts_local'].fillna(pd.NA)

    # 10) compute ratio for sorting: max_value_in_window / global_P95 (handle zeros/NaN)
    def safe_factor(maxv, p95):
        try:
            if pd.isna(maxv) or pd.isna(p95) or float(p95) == 0:
                return np.nan
            return float(maxv) / float(p95)
        except Exception:
            return np.nan

    summary['factor_over_p95'] = summary.apply(lambda r: safe_factor(r['max_value_in_window'], r['global_P95']), axis=1)

    # 11) sort by anomalies_in_window_count desc, then factor desc
    result_df = (
        summary[
            ['cmdb_id','trace_name','global_P95','total_points_in_full_series',
             'total_points_in_window','anomalies_in_window_count',
             'earliest_anomaly_ts_local','max_value_in_window','factor_over_p95']
        ]
        .sort_values(by=['anomalies_in_window_count','factor_over_p95'], ascending=[False, False])
        .reset_index(drop=True)
        .head(20)
    )

    # Round numeric columns for compactness
    result_df['global_P95'] = result_df['global_P95'].round(6)
    result_df['max_value_in_window'] = result_df['max_value_in_window'].round(6)
    result_df['factor_over_p95'] = result_df['factor_over_p95'].round(6)

    # Final output: keep requested columns (exclude helper factor_over_p95 if not needed)
    final_df = result_df[[
        'cmdb_id','trace_name','global_P95','total_points_in_full_series',
        'total_points_in_window','anomalies_in_window_count','earliest_anomaly_ts_local','max_value_in_window'
    ]].copy()

# Display compact final table (top 20)
final_df

```
Out[5]:
```
summary_text = (
    "Scan result for trace_name containing 'mysql' or 'redis' (case-insensitive):\n"
    "- No matching trace_name values were found in trace.csv.\n"
    "- Therefore no global P95 or window-anomaly rows exist for MySQL/Redis traces.\n"
    "- Incident window: 2021-03-10 16:30:00 to 2021-03-10 17:00:00 (UTC+8).\n"
    "- Interpretation: trace.csv contains no trace-level entries referencing MySQL or Redis for this dataset, "
    "so there is no trace-based evidence (in this file) of MySQL/Redis anomalies during the incident window."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, global_P95, total_points_in_full_series, total_points_in_window, anomalies_in_window_count, earliest_anomaly_ts_local, max_value_in_window]
Index: []```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import datetime as dt
import numpy as np

# Reuse existing log_df and timezone/window variables if present; otherwise load/define
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

tz = pytz.timezone('Asia/Shanghai')
try:
    win_start_epoch, win_end_epoch
except NameError:
    win_start = tz.localize(dt.datetime(2021,3,10,16,30,0))
    win_end   = tz.localize(dt.datetime(2021,3,10,17,0,0))
    win_start_epoch = int(win_start.timestamp())
    win_end_epoch = int(win_end.timestamp())

# Ensure correct dtypes
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

group_keys = ['cmdb_id','log_name']

# 1) compute global P95 per (cmdb_id, log_name) using full series (no filtering)
global_p95 = (
    log_df
    .groupby(group_keys, observed=True)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# 2) total points in full series per group
total_full = (
    log_df
    .groupby(group_keys, observed=True)
    .size()
    .reset_index(name='total_points_in_full_series')
)

# 3) filter to incident window (inclusive)
window_df = log_df[(log_df['timestamp'] >= win_start_epoch) & (log_df['timestamp'] <= win_end_epoch)].copy()

# 4) total points in window per group
total_window = (
    window_df
    .groupby(group_keys, observed=True)
    .size()
    .reset_index(name='total_points_in_window')
)

# 5) identify anomalies in window: value > global_P95
win_with_p95 = window_df.merge(global_p95, on=group_keys, how='left')
anomalies = win_with_p95[win_with_p95['value'] > win_with_p95['global_P95']].copy()

# 6) aggregate anomalies: count and earliest timestamp
anom_agg = (
    anomalies
    .groupby(group_keys, observed=True)
    .agg(
        anomalies_in_window_count=('value','size'),
        earliest_anom_ts_epoch=('timestamp','min')
    )
    .reset_index()
)

# convert earliest epoch to localized string
def epoch_to_local_str(e):
    if pd.isna(e):
        return pd.NA
    try:
        return dt.datetime.fromtimestamp(int(e), tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return pd.NA

if not anom_agg.empty:
    anom_agg['earliest_anomaly_ts_local'] = anom_agg['earliest_anom_ts_epoch'].apply(epoch_to_local_str)
else:
    anom_agg['earliest_anomaly_ts_local'] = pd.Series(dtype="object")

# 7) max value in window per group
max_in_window = (
    window_df
    .groupby(group_keys, observed=True)['value']
    .max()
    .reset_index(name='max_value_in_window')
)

# 8) assemble summary table
summary = global_p95.merge(total_full, on=group_keys, how='left')
summary = summary.merge(total_window, on=group_keys, how='left')
summary = summary.merge(anom_agg[[*group_keys, 'anomalies_in_window_count', 'earliest_anomaly_ts_local']], on=group_keys, how='left')
summary = summary.merge(max_in_window, on=group_keys, how='left')

# fill NaNs for counts
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['anomalies_in_window_count'] = summary['anomalies_in_window_count'].fillna(0).astype(int)
summary['earliest_anomaly_ts_local'] = summary['earliest_anomaly_ts_local'].fillna(pd.NA)

# 9) compute factor for tie-breaker sorting: max_value_in_window / global_P95
def safe_factor(maxv, p95):
    try:
        if pd.isna(maxv) or pd.isna(p95) or float(p95) == 0:
            return np.nan
        return float(maxv) / float(p95)
    except Exception:
        return np.nan

summary['factor_over_p95'] = summary.apply(lambda r: safe_factor(r['max_value_in_window'], r['global_P95']), axis=1)

# 10) final selection and sorting
result_df = (
    summary[
        ['cmdb_id','log_name','global_P95','total_points_in_full_series',
         'total_points_in_window','anomalies_in_window_count',
         'earliest_anomaly_ts_local','max_value_in_window','factor_over_p95']
    ]
    .sort_values(by=['anomalies_in_window_count','factor_over_p95'], ascending=[False, False])
    .reset_index(drop=True)
    .head(20)
)

# Round numeric columns for compact display
result_df['global_P95'] = result_df['global_P95'].round(6)
result_df['max_value_in_window'] = result_df['max_value_in_window'].round(6)
result_df['factor_over_p95'] = result_df['factor_over_p95'].round(6)

# Final output: requested columns (exclude helper factor if not required)
final_df = result_df[[
    'cmdb_id','log_name','global_P95','total_points_in_full_series',
    'total_points_in_window','anomalies_in_window_count','earliest_anomaly_ts_local','max_value_in_window'
]].copy()

final_df

```
Out[6]:
```
summary_text = (
    "Log anomaly summary for incident window 2021-03-10 16:30–17:00 (UTC+8):\n"
    "- Primary findings:\n"
    "  * Several services show short spikes in log.row_count above their global P95 (each with 2 anomaly points in the window):\n"
    "    - Mysql01, Tomcat04, apache01, Tomcat02, apache02, Tomcat03, Tomcat01 (all have log.row_count spikes).\n"
    "    - Max row_count values observed in-window range from ~491 to ~981 depending on component.\n"
    "  * Tomcat GC pause spikes were observed (log.gc_pause_max): Tomcat03 (2 anomaly points, earliest 16:35),\n"
    "    Tomcat01 and Tomcat02 show smaller single-point GC anomalies.\n"
    "- Counts and timing:\n"
    "  * Eight (cmdb_id,log_name) pairs have 2 anomalous points in the window; two pairs have 1 anomalous point.\n"
    "  * Earliest anomaly timestamps vary between ~16:31 and ~16:49 (UTC+8) depending on component.\n"
    "- Other notes:\n"
    "  * log.error_count is 0 for most components (no error-count spikes detected in this window).\n"
    "- Interpretation / next steps:\n"
    "  * The widespread log.row_count spikes across multiple app servers and Apache suggest a throughput or request-rate increase\n"
    "    (or bursty processing) during the incident window rather than an obvious surge in logged errors.\n"
    "  * GC pause spikes on Tomcat instances are present but small; investigate whether GC impact correlates with the row_count spikes.\n"
    "  * Recommended: correlate these log.row_count spikes with upstream request rates, application metrics (CPU/memory),\n"
    "    and downstream DB activity (MySQL) to determine whether this was legitimate traffic, a queue/backlog being drained, or an anomaly."
)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  global_P95  total_points_in_full_series  total_points_in_window  anomalies_in_window_count earliest_anomaly_ts_local  max_value_in_window
0    Mysql01     log.row_count      8.6500                           28                      28                          2   2021-03-10 16:37:00 CST                10.00
1   Tomcat03  log.gc_pause_max      0.0895                           22                      22                          2   2021-03-10 16:35:00 CST                 0.10
2   Tomcat04     log.row_count    491.0000                           31                      31                          2   2021-03-10 16:31:00 CST               521.00
3   apache01     log.row_count    940.5000                           31                      31                          2   2021-03-10 16:49:00 CST               981.00
4   Tomcat02     log.row_count    506.5000                           31                      31                          2   2021-03-10 16:36:00 CST               527.00
5   apache02     log.row_count    951.5000                           31                      31                          2   2021-03-10 16:36:00 CST               978.00
6   Tomcat03     log.row_count    479.0000                           31                      31                          2   2021-03-10 16:49:00 CST               491.00
7   Tomcat01     log.row_count    484.6500                           28                      28                          2   2021-03-10 16:47:00 CST               491.00
8   Tomcat01  log.gc_pause_max      0.0930                           19                      19                          1   2021-03-10 16:46:00 CST                 0.12
9   Tomcat02  log.gc_pause_max      0.0900                           30                      30                          1   2021-03-10 16:56:00 CST                 0.10
10   Mysql02     log.row_count      1.0000                            1                       1                          0                      <NA>                 1.00
11  Tomcat04  log.gc_pause_max      0.0800                           21                      21                          0                      <NA>                 0.08
12   Mysql01   log.error_count      0.0000                           28                      28                          0                      <NA>                 0.00
13   Mysql02   log.error_count      0.0000                            1                       1                          0                      <NA>                 0.00
14  Tomcat01   log.error_count      0.0000                           28                      28                          0                      <NA>                 0.00
15  Tomcat02   log.error_count      0.0000                           31                      31                          0                      <NA>                 0.00
16  Tomcat03   log.error_count      0.0000                           31                      31                          0                      <NA>                 0.00
17  Tomcat04   log.error_count      0.0000                           31                      31                          0                      <NA>                 0.00
18  apache01   log.error_count      0.0000                           31                      31                          0                      <NA>                 0.00
19  apache02   log.error_count      0.0000                           31                      31                          0                      <NA>                 0.00```
```